# Phase 5 — Final Evaluation and Model Comparison

This notebook performs the final evaluation phase for the aerial object classification pipeline.

## Objectives
- Evaluate the held-out **test set** on:
  - Custom CNN
  - Best transfer-learning model
  - Optional second transfer-learning model
- Generate evaluation artifacts:
  - confusion matrices
  - classification reports
  - test accuracy, precision, recall, F1-score
  - comparison table
  - model comparison bar plot
  - hard-image error analysis grid
- Compare models on:
  - test performance
  - training time
  - generalization behavior
  - model size
  - suitability for Streamlit deployment
- Select the final classification model and export:
  - `models/classification/final/best_classifier.keras`
  - `models/classification/final/class_mapping.json`
  - `models/classification/final/final_metrics.json`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from pathlib import Path
import shutil
import sys

import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path("/content/drive/MyDrive/Aerial_Object_Classification_Detection")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.config import load_yaml
from src.utils.io import load_json, save_text
from src.utils.paths import load_paths_config, ensure_dir
from src.utils.seed import set_seed

from src.modeling.train_classifier import build_tf_directory_datasets

from src.evaluation.evaluate_classifier import (
    evaluate_keras_classifier_on_paths,
    build_evaluated_model_payload,
    export_final_selected_classifier,
    run_full_evaluation_suite,
)
from src.evaluation.confusion_matrix_plot import save_confusion_matrix_figure
from src.evaluation.classification_report_utils import save_classification_artifacts

## 1. Load configs and define project paths

This section loads the shared configuration files and prepares all directories and output paths required by the evaluation phase.

In [5]:
paths_config = load_paths_config(PROJECT_ROOT / "configs" / "paths.yaml")
classification_config = load_yaml(PROJECT_ROOT / "configs" / "classification_config.yaml")
transfer_learning_config = load_yaml(PROJECT_ROOT / "configs" / "transfer_learning_config.yaml")

set_seed(42)

processed_classification_root = paths_config.get(
    "processed_classification_root",
    PROJECT_ROOT / "data" / "processed" / "classification",
)

reports_dir = PROJECT_ROOT / "reports"
figures_eval_dir = PROJECT_ROOT / "figures" / "evaluation"
docs_images_dir = PROJECT_ROOT / "docs" / "images"
docs_dir = PROJECT_ROOT / "docs"
models_final_dir = PROJECT_ROOT / "models" / "classification" / "final"

ensure_dir(reports_dir)
ensure_dir(figures_eval_dir)
ensure_dir(docs_images_dir)
ensure_dir(docs_dir)
ensure_dir(models_final_dir)

evaluation_report_path = reports_dir / "evaluation_report.md"
comparison_csv_path = reports_dir / "model_comparison.csv"
comparison_plot_path = figures_eval_dir / "model_comparison_barplot.png"
error_analysis_grid_path = figures_eval_dir / "error_analysis_grid.png"

docs_confusion_matrix_path = docs_images_dir / "confusion_matrix.png"
docs_custom_curves_path = docs_images_dir / "custom_cnn_curves.png"
docs_model_notes_path = docs_dir / "model_comparison_notes.md"

final_best_model_path = models_final_dir / "best_classifier.keras"
final_class_mapping_path = models_final_dir / "class_mapping.json"
final_metrics_path = models_final_dir / "final_metrics.json"

## 2. Resolve saved training artifacts

This step locates the saved `.keras` models and their corresponding metrics JSON files for:
- custom CNN
- all enabled transfer-learning backbones

In [6]:
custom_cnn_dir = PROJECT_ROOT / classification_config["custom_cnn"]["artifacts"]["model_dir"]
custom_cnn_best_model_path = custom_cnn_dir / classification_config["custom_cnn"]["artifacts"]["best_model_name"]
custom_cnn_history_path = custom_cnn_dir / classification_config["custom_cnn"]["artifacts"]["history_name"]
custom_cnn_metrics_path = custom_cnn_dir / classification_config["custom_cnn"]["artifacts"]["metrics_name"]

transfer_registry = transfer_learning_config["models"]

artifact_rows = []

if custom_cnn_metrics_path.exists() and custom_cnn_best_model_path.exists():
    custom_metrics_saved = load_json(custom_cnn_metrics_path)
    artifact_rows.append(
        {
            "model_name": "custom_cnn",
            "family": "custom_cnn",
            "backbone": "custom_cnn",
            "model_path": str(custom_cnn_best_model_path),
            "metrics_path": str(custom_cnn_metrics_path),
            "val_accuracy": custom_metrics_saved.get("validation", {}).get("accuracy"),
            "training_time_seconds": custom_metrics_saved.get("training_time_seconds"),
        }
    )

for model_name, model_cfg in transfer_registry.items():
    if not model_cfg.get("enabled", False):
        continue

    model_dir = PROJECT_ROOT / model_cfg["artifacts"]["model_dir"]
    metrics_path = model_dir / model_cfg["artifacts"]["metrics_name"]
    best_model_path = model_dir / model_cfg["artifacts"]["best_model_name"]

    if metrics_path.exists() and best_model_path.exists():
        saved_metrics = load_json(metrics_path)
        artifact_rows.append(
            {
                "model_name": model_name,
                "family": "transfer_learning",
                "backbone": saved_metrics.get("backbone", model_cfg["backbone"]),
                "model_path": str(best_model_path),
                "metrics_path": str(metrics_path),
                "val_accuracy": saved_metrics.get("validation", {}).get("accuracy"),
                "training_time_seconds": saved_metrics.get("training_time_seconds"),
            }
        )

artifact_df = pd.DataFrame(artifact_rows)

if artifact_df.empty:
    raise FileNotFoundError("No saved classifier artifacts were found for evaluation.")

artifact_df

,model_name,family,backbone,model_path,metrics_path,val_accuracy,training_time_seconds
0,custom_cnn,custom_cnn,custom_cnn,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.780543,NaN
1,mobilenet,transfer_learning,mobilenetv2,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.945701,1695.236906
2,resnet50,transfer_learning,resnet50,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.966063,8201.833738
3,efficientnetb0,transfer_learning,efficientnetb0,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.986425,4291.913510


## 3. Select evaluation candidates

The final evaluation includes:
- `custom_cnn`
- the best transfer-learning model
- an optional second transfer-learning model if present

In [7]:
transfer_candidates_df = artifact_df.loc[artifact_df["family"] == "transfer_learning"].copy()

if transfer_candidates_df.empty:
    raise ValueError("No saved transfer-learning artifacts were found.")

transfer_candidates_df = transfer_candidates_df.sort_values(
    by=["val_accuracy", "training_time_seconds"],
    ascending=[False, True],
).reset_index(drop=True)

best_transfer_name = transfer_candidates_df.iloc[0]["model_name"]
second_transfer_name = None

if len(transfer_candidates_df) > 1:
    second_transfer_name = transfer_candidates_df.iloc[1]["model_name"]

selected_model_names = ["custom_cnn", best_transfer_name]
if second_transfer_name is not None and second_transfer_name not in selected_model_names:
    selected_model_names.append(second_transfer_name)

selected_models_df = artifact_df.loc[artifact_df["model_name"].isin(selected_model_names)].copy()
selected_models_df = selected_models_df.reset_index(drop=True)

selected_models_df

,model_name,family,backbone,model_path,metrics_path,val_accuracy,training_time_seconds
0,custom_cnn,custom_cnn,custom_cnn,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.780543,NaN
1,resnet50,transfer_learning,resnet50,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.966063,8201.833738
2,efficientnetb0,transfer_learning,efficientnetb0,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.986425,4291.913510


## 4. Build the test-set inventory and class mappings

This step discovers:
- the test image paths
- the deterministic class order
- `class_to_index` and `index_to_class` mappings for evaluation and export

In [8]:
raw_datasets = build_tf_directory_datasets(
    dataset_root=processed_classification_root,
    image_size=classification_config["custom_cnn"]["training"]["image_size"],
    batch_size=classification_config["custom_cnn"]["training"]["batch_size"],
    label_mode=classification_config["custom_cnn"]["training"]["label_mode"],
    seed=classification_config["custom_cnn"]["training"]["seed"],
)

class_names = list(raw_datasets["class_names"])
class_to_index = {class_name: idx for idx, class_name in enumerate(class_names)}
index_to_class = {idx: class_name for class_name, idx in class_to_index.items()}

test_image_paths = []
test_true_names = []

for class_name in class_names:
    class_dir = processed_classification_root / "test" / class_name
    for image_path in sorted(class_dir.glob("*.jpg")):
        test_image_paths.append(image_path)
        test_true_names.append(class_name)

if len(test_image_paths) == 0:
    raise ValueError("No test images found in processed classification test split.")

y_true = np.array([class_to_index[name] for name in test_true_names], dtype=int)

len(test_image_paths), class_names

Found 2662 files belonging to 2 classes.
Found 442 files belonging to 2 classes.
Found 215 files belonging to 2 classes.


(215, ['bird', 'drone'])

## 5. Evaluate each selected model on the test set

This section uses the reusable evaluation module to:
- load each saved classifier
- apply the correct preprocessing branch
- produce fresh test-set predictions
- compute evaluation metrics
- build model payloads for comparison

In [9]:
evaluated_model_payloads = []

for _, row in selected_models_df.iterrows():
    model_name = row["model_name"]
    backbone = row["backbone"]
    model_path = Path(row["model_path"])
    metrics_path = Path(row["metrics_path"])

    saved_metrics = load_json(metrics_path) if metrics_path.exists() else {}

    if model_name == "custom_cnn":
        preprocess_mode = "custom"
        speed_bucket = "medium"
        deployment_note = "Light-to-moderate size custom model; suitable if accuracy is competitive."
    else:
        preprocess_mode = "transfer"

        backbone_lower = str(backbone).lower()
        if backbone_lower in {"mobilenet", "mobilenetv2"}:
            speed_bucket = "fast"
            deployment_note = "Best deployment fit for Streamlit among transfer models."
        elif backbone_lower == "efficientnetb0":
            speed_bucket = "medium"
            deployment_note = "Good balance between efficiency and accuracy."
        elif backbone_lower == "resnet50":
            speed_bucket = "slower"
            deployment_note = "Heavier backbone; good accuracy potential but slower deployment."
        else:
            speed_bucket = "unknown"
            deployment_note = "Deployment suitability should be validated."

    eval_output = evaluate_keras_classifier_on_paths(
        model_path=model_path,
        image_paths=test_image_paths,
        y_true=y_true,
        class_names=class_names,
        image_size=classification_config["custom_cnn"]["training"]["image_size"],
        preprocess_mode=preprocess_mode,
        backbone_name=None if preprocess_mode == "custom" else backbone,
    )

    payload = build_evaluated_model_payload(
        model_name=model_name,
        backbone=backbone,
        model_path=model_path,
        saved_metrics_path=metrics_path,
        eval_output=eval_output,
        class_names=class_names,
        val_accuracy=saved_metrics.get("validation", {}).get("accuracy"),
        training_time_seconds=saved_metrics.get("training_time_seconds"),
        speed_bucket=speed_bucket,
        deployment_note=deployment_note,
    )

    payload["y_true"] = eval_output["y_true"]
    payload["y_pred"] = eval_output["y_pred"]
    payload["y_score"] = eval_output["y_score"]

    evaluated_model_payloads.append(payload)

len(evaluated_model_payloads)

3

## 6. Save per-model confusion matrices and classification artifacts

For each evaluated model, this section creates:
- confusion matrix image
- metrics JSON
- markdown evaluation summary
- optional classification report CSV

In [12]:
per_model_eval_dir = reports_dir / "per_model"
ensure_dir(per_model_eval_dir)

confusion_matrix_paths = {}

for payload in evaluated_model_payloads:
    model_name = payload["model_name"]

    cm_path = figures_eval_dir / f"{model_name}_confusion_matrix.png"
    save_confusion_matrix_figure(
        confusion_matrix=payload["confusion_matrix"],
        class_names=class_names,
        output_path=cm_path,
        title=f"{model_name} — Test Confusion Matrix",
    )
    confusion_matrix_paths[model_name] = cm_path

    metrics_json_path = per_model_eval_dir / f"{model_name}_evaluation_metrics.json"
    report_md_path = per_model_eval_dir / f"{model_name}_evaluation_summary.md"
    report_csv_path = per_model_eval_dir / f"{model_name}_classification_report.csv"

    payload["confusion_matrix_path"] = str(cm_path)

    curves_paths = []
    if model_name == "custom_cnn":
        acc_curve = PROJECT_ROOT / classification_config["custom_cnn"]["artifacts"]["accuracy_curve_path"]
        loss_curve = PROJECT_ROOT / classification_config["custom_cnn"]["artifacts"]["loss_curve_path"]
        if acc_curve.exists():
            curves_paths.append(str(acc_curve))
        if loss_curve.exists():
            curves_paths.append(str(loss_curve))
    else:
        model_cfg = transfer_registry[model_name]
        acc_curve = PROJECT_ROOT / model_cfg["artifacts"]["accuracy_curve_path"]
        loss_curve = PROJECT_ROOT / model_cfg["artifacts"]["loss_curve_path"]
        if acc_curve.exists():
            curves_paths.append(str(acc_curve))
        if loss_curve.exists():
            curves_paths.append(str(loss_curve))

    payload["curves_paths"] = curves_paths

    payload_for_disk = {
        key: value for key, value in payload.items()
        if key not in {"y_true", "y_pred", "y_score"}
    }

    save_classification_artifacts(
        metrics_payload=payload_for_disk,
        metrics_json_path=metrics_json_path,
        report_md_path=report_md_path,
        classification_report_csv_path=report_csv_path,
    )

confusion_matrix_paths

{'custom_cnn': PosixPath('/content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/evaluation/custom_cnn_confusion_matrix.png'),
 'resnet50': PosixPath('/content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/evaluation/resnet50_confusion_matrix.png'),
 'efficientnetb0': PosixPath('/content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/evaluation/efficientnetb0_confusion_matrix.png')}

## 7. Run full model comparison and error analysis

This section:
- builds the model comparison table
- saves the comparison CSV
- saves the comparison bar plot
- selects the best model
- performs binary error analysis
- saves the hard-image error-analysis grid

In [13]:
comparison_ready_payloads = []

for payload in evaluated_model_payloads:
    row_payload = {
        key: value for key, value in payload.items()
        if key not in {"y_true", "y_pred", "y_score"}
    }
    row_payload["y_pred"] = payload["y_pred"]
    row_payload["y_score"] = payload["y_score"]
    comparison_ready_payloads.append(row_payload)

evaluation_suite_outputs = run_full_evaluation_suite(
    evaluated_model_payloads=comparison_ready_payloads,
    image_paths=test_image_paths,
    y_true=y_true,
    class_names=class_names,
    index_to_class=index_to_class,
    comparison_csv_path=comparison_csv_path,
    comparison_plot_path=comparison_plot_path,
    error_analysis_grid_path=error_analysis_grid_path,
)

comparison_df = evaluation_suite_outputs["comparison_df"]
selected_row = evaluation_suite_outputs["selected_row"]
prediction_df = evaluation_suite_outputs["prediction_df"]
error_tables = evaluation_suite_outputs["error_tables"]
error_summary = evaluation_suite_outputs["error_summary"]

comparison_df

,model_name,backbone,model_path,metrics_path,accuracy,precision,recall,f1_score,val_accuracy,generalization_gap_abs,...,deployment_note,confusion_matrix,classification_report,class_names,confusion_matrix_path,curves_paths,y_pred,y_score,streamlit_suitability,selection_score
0,resnet50,resnet50,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.986047,1.000000,0.968085,0.983784,0.966063,0.019983,...,Heavier backbone; good accuracy potential but ...,"[[121, 0], [3, 91]]","{'bird': {'precision': 0.9758064516129032, 're...","[bird, drone]",/content/drive/MyDrive/Aerial_Object_Classific...,[/content/drive/MyDrive/Aerial_Object_Classifi...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0005538988625630736, 0.00015237598563544452...",limited,0.984835
1,efficientnetb0,efficientnetb0,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.972093,0.978261,0.957447,0.967742,0.986425,0.014332,...,Good balance between efficiency and accuracy.,"[[119, 2], [4, 90]]","{'bird': {'precision': 0.967479674796748, 'rec...","[bird, drone]",/content/drive/MyDrive/Aerial_Object_Classific...,[/content/drive/MyDrive/Aerial_Object_Classifi...,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.9981860518455505, 5.497036181623116e-05, 5....",good,0.971171
2,custom_cnn,custom_cnn,/content/drive/MyDrive/Aerial_Object_Classific...,/content/drive/MyDrive/Aerial_Object_Classific...,0.827907,0.827586,0.765957,0.795580,0.780543,0.047364,...,Light-to-moderate size custom model; suitable ...,"[[106, 15], [22, 72]]","{'bird': {'precision': 0.828125, 'recall': 0.8...","[bird, drone]",/content/drive/MyDrive/Aerial_Object_Classific...,[/content/drive/MyDrive/Aerial_Object_Classifi...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ...","[0.097664013504982, 0.009144741110503674, 0.00...",good,0.821206


## 8. Review false positives, false negatives, and hard images

This section helps inspect the error distribution and common confusion patterns of the final selected model.

In [14]:
false_positive_df = error_tables["false_positives"].copy()
false_negative_df = error_tables["false_negatives"].copy()
hard_images_df = error_tables["hard_images"].copy()

fp_count = len(false_positive_df)
fn_count = len(false_negative_df)
hard_count = len(hard_images_df)

print("Selected model:", selected_row["model_name"])
print("False positives:", fp_count)
print("False negatives:", fn_count)
print("Hard images:", hard_count)
print("Summary:", error_summary)

Selected model: resnet50
False positives: 0
False negatives: 3
Hard images: 3
Summary: False positives: 0, false negatives: 3, total hard images: 3. The current error distribution shows more false negatives than false positives.


## 9. Export the final selected classifier

The best-ranked model is exported into the final deployment directory together with:
- final class mapping
- final metrics JSON

In [17]:
comparison_export_df = comparison_df.copy()
comparison_export_df = comparison_export_df.drop(
    columns=[col for col in ["y_true", "y_pred", "y_score"] if col in comparison_export_df.columns],
    errors="ignore",
)

final_export_payload = export_final_selected_classifier(
    selected_row=selected_row,
    final_model_path=final_best_model_path,
    final_class_mapping_path=final_class_mapping_path,
    final_metrics_path=final_metrics_path,
    class_to_index=class_to_index,
    index_to_class=index_to_class,
    all_comparisons_df=comparison_export_df,
)

## 10. Copy documentation-ready figures

This step copies the selected model’s confusion matrix into the documentation image folder and prepares the custom CNN curves copy for documentation use.

In [19]:
selected_model_name = selected_row["model_name"]

selected_cm_path = confusion_matrix_paths[selected_model_name]
shutil.copy2(selected_cm_path, docs_confusion_matrix_path)

custom_accuracy_curve_path = PROJECT_ROOT / classification_config["custom_cnn"]["artifacts"]["accuracy_curve_path"]
custom_loss_curve_path = PROJECT_ROOT / classification_config["custom_cnn"]["artifacts"]["loss_curve_path"]

if custom_accuracy_curve_path.exists():
    shutil.copy2(custom_accuracy_curve_path, docs_images_dir / custom_accuracy_curve_path.name)

if custom_loss_curve_path.exists():
    shutil.copy2(custom_loss_curve_path, docs_images_dir / custom_loss_curve_path.name)

## 11. Write evaluation report and model comparison notes

This final documentation step writes:
- `reports/evaluation_report.md`
- `docs/model_comparison_notes.md`

In [20]:
report_lines = [
    "# Final Evaluation Report",
    "",
    "## Evaluated Models",
    "",
]

for _, row in comparison_df.iterrows():
    training_time_text = "N/A" if pd.isna(row["training_time_seconds"]) else f"{row['training_time_seconds']:.2f}"
    generalization_gap_text = "N/A" if pd.isna(row["generalization_gap_abs"]) else f"{row['generalization_gap_abs']:.4f}"

    report_lines.extend([
        f"### {row['model_name']}",
        f"- Backbone: {row['backbone']}",
        f"- Test accuracy: {row['accuracy']:.4f}",
        f"- Precision: {row['precision']:.4f}",
        f"- Recall: {row['recall']:.4f}",
        f"- F1-score: {row['f1_score']:.4f}",
        f"- Model size (MB): {row['model_size_mb']:.3f}",
        f"- Training time (seconds): {training_time_text}",
        f"- Generalization gap |val_acc - test_acc|: {generalization_gap_text}",
        f"- Streamlit suitability: {row['streamlit_suitability']}",
        f"- Deployment note: {row['deployment_note']}",
        "",
    ])

report_lines.extend([
    "## Error Analysis",
    "",
    f"- Final selected model: **{selected_model_name}**",
    f"- False positives: **{fp_count}**",
    f"- False negatives: **{fn_count}**",
    f"- Total hard images: **{hard_count}**",
    f"- Summary: {error_summary}",
    "",
    "Common confusion should be interpreted from the error-analysis grid with attention to object size, sky clutter, silhouette ambiguity, distance, and motion blur.",
    "",
    "## Final Selection",
    "",
    f"- Selected classifier: **{selected_model_name}**",
    f"- Exported model path: `{final_best_model_path}`",
    f"- Class mapping path: `{final_class_mapping_path}`",
    f"- Final metrics path: `{final_metrics_path}`",
    "",
    "## Comparison Table",
    "",
    comparison_df.to_markdown(index=False),
    "",
])

save_text("\n".join(report_lines), evaluation_report_path)

notes_lines = [
    "# Model Comparison Notes",
    "",
    f"- Best transfer model selected for final evaluation: **{best_transfer_name}**",
    f"- Optional second transfer model included: **{second_transfer_name if second_transfer_name is not None else 'None'}**",
    f"- Final selected classifier after full test-set evaluation: **{selected_model_name}**",
    "",
    "## Selection Criteria",
    "",
    "- best balanced performance on the held-out test set",
    "- stable generalization relative to validation performance",
    "- manageable model size",
    "- practical inference suitability for Streamlit deployment",
    "",
    "## Practical Deployment Recommendation",
    "",
    f"- Recommended deployment classifier: **{selected_model_name}**",
    f"- Reason: {selected_row['deployment_note']}",
]

save_text("\n".join(notes_lines), docs_model_notes_path)

## 12. Final artifact summary

This cell prints the final output paths created in this notebook.

In [21]:
artifact_summary = {
    "evaluation_report": str(evaluation_report_path),
    "comparison_csv": str(comparison_csv_path),
    "comparison_plot": str(comparison_plot_path),
    "error_analysis_grid": str(error_analysis_grid_path),
    "docs_confusion_matrix": str(docs_confusion_matrix_path),
    "docs_model_notes": str(docs_model_notes_path),
    "final_best_model": str(final_best_model_path),
    "final_class_mapping": str(final_class_mapping_path),
    "final_metrics_json": str(final_metrics_path),
}

artifact_summary

{'evaluation_report': '/content/drive/MyDrive/Aerial_Object_Classification_Detection/reports/evaluation_report.md',
 'comparison_csv': '/content/drive/MyDrive/Aerial_Object_Classification_Detection/reports/model_comparison.csv',
 'comparison_plot': '/content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/evaluation/model_comparison_barplot.png',
 'error_analysis_grid': '/content/drive/MyDrive/Aerial_Object_Classification_Detection/figures/evaluation/error_analysis_grid.png',
 'docs_confusion_matrix': '/content/drive/MyDrive/Aerial_Object_Classification_Detection/docs/images/confusion_matrix.png',
 'docs_model_notes': '/content/drive/MyDrive/Aerial_Object_Classification_Detection/docs/model_comparison_notes.md',
 'final_best_model': '/content/drive/MyDrive/Aerial_Object_Classification_Detection/models/classification/final/best_classifier.keras',
 'final_class_mapping': '/content/drive/MyDrive/Aerial_Object_Classification_Detection/models/classification/final/class_mapping.